# 🪷 CHÙA LINH ỨNG ONLINE — TIKTOK LIVE INTERACTIVE 8K
### HỆ THỐNG PHÁT SÓNG TƯƠNG TÁC TÂM LINH THỜI GIAN THỰC TRÊN GOOGLE COLAB

Hệ thống phát sóng tương tác chuẩn khung hình dọc **9:16 (1080x1920)** với đồ họa Chánh Điện Đức Phật A Di Đà dát vàng 8K sắc nét, hỗ trợ 2 hình thức phát sóng:
- **Cách 1 (Khuyên dùng - Nhẹ máy nhất):** Colab làm máy chủ -> Cấp link HTTPS Cloudflare Tunnel để dán vào mục **Liên kết** trong **TikTok LIVE Studio** hoặc **OBS Studio** (Không bao giờ bị lỗi màn hình đen hay bắt đăng nhập).
- **Cách 2 (Treo máy tự động 24/7):** Colab tự tạo Màn hình ảo (Xvfb 1080x1920) và dùng FFmpeg đẩy luồng trực tiếp lên TikTok bằng **Stream Key** mà không cần bật máy tính cá nhân!


In [ ]:
# @title 🚀 BƯỚC 1: Khởi Chạy Server Chùa Linh Ứng 8K & Lấy Link Dán Vào TikTok LIVE Studio { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi động toàn bộ hệ thống:

TikTok_Username = "@2166541467" # @param {type:"string"}
Bat_Giong_Doc_AI = True # @param {type:"boolean"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/4] Đang đồng bộ mã nguồn Chùa Online 8K...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app

print("⏳ [2/4] Đang kiểm tra thư viện Node.js & Cloudflare Tunnel...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Cài đặt ZeroTTS nếu chọn bật giọng đọc AI
if Bat_Giong_Doc_AI:
    try:
        import zerotts
    except ImportError:
        print("🎙️ Đang cài đặt thư viện giọng đọc AI ZeroTTS tiếng Việt...")
        !pip install --quiet zerotts pillow

print("⏳ [3/4] Đang dọn dẹp tiến trình cũ và khởi chạy Server...")
!pkill -9 -f "server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server.js ở chế độ nền
with open("/tmp/server.log", "w") as f_log:
    server_proc = subprocess.Popen(["node", "server.js"], cwd="/content/app", stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

# Khởi chạy Cloudflare Tunnel công khai
print("⏳ [4/4] Đang khởi tạo đường truyền HTTPS công khai...")
with open("/tmp/cloudflared.log", "w") as cf_log:
    tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    obs_link = f"{public_url}/?obs=1"
    admin_link = f"{public_url}/admin"
    html_card = f"""
    <div style="background: linear-gradient(135deg, #2b0805 0%, #150302 100%); border: 2px solid #e5a93c; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(229,169,60,0.3);">
        <h2 style="color: #ffd700; margin-top: 0; font-size: 22px;">🎉 CHÙA LINH ỨNG ONLINE 8K ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #f5deb3;">
            Đường truyền công khai đã được thiết lập qua Cloudflare Tunnel (<b>Không bị hỏi đăng nhập GitHub, không bị lỗi màn hình đen</b>).
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{obs_link}" target="_blank" style="background: linear-gradient(180deg, #d97706 0%, #b45309 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #ffd700; box-shadow: 0 4px 15px rgba(217,119,6,0.4);">
                📺 [1] MỞ MÀN HÌNH LIVE (Dán link này vào TikTok LIVE Studio)
            </a>
            <a href="{admin_link}" target="_blank" style="background: linear-gradient(180deg, #15803d 0%, #166534 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #86efac;">
                🎛️ [2] MỞ TRANG QUẢN TRỊ & THỬ NGHIỆM NÚT BẤM
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.4); border-left: 4px solid #ffd700; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #ffe4b5;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{obs_link}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/cloudflared.log


### 📺 HƯỚNG DẪN CÀI ĐẶT VÀO TIKTOK LIVE STUDIO (CHỈ MẤT 30 GIÂY)
1. Mở phần mềm **TikTok LIVE Studio** trên máy tính của bạn.
2. Bấm vào nút **"Thêm nguồn"** $\rightarrow$ chọn mục **"Liên kết"** (biểu tượng hình quả cầu 🌐).
3. Dán đường link `https://xxxx.trycloudflare.com/?obs=1` (được sinh ra ở Bước 1 ở trên).
4. Đặt độ phân giải là: **`1080 x 1920`** (chuẩn khung dọc 9:16 của TikTok).
5. Tích chọn **"Bật âm thanh"** để khán giả nghe được tiếng mõ tụng kinh và đại hồng chung.
6. Bấm **Bắt đầu phát LIVE**, sau đó mở link Quản trị `[2]` để kết nối tài khoản TikTok của bạn!


---
## 🔴 PHẦN 2: TỰ ĐỘNG PHÁT LIVESTREAM LÊN TIKTOK TỪ COLAB (KHÔNG CẦN BẬT MÁY TÍNH)
Phần này sẽ khởi tạo **màn hình ảo 1080x1920** ngay bên trong Google Colab, mở Chánh Điện Chùa Linh Ứng và dùng **FFmpeg** đẩy thẳng luồng video 60FPS + âm thanh chuông mõ lên kênh TikTok của bạn bằng **Stream Key**!


In [ ]:
# @title 🎥 BƯỚC 2: Nhập Stream Key & Khởi Chạy Livestream Tự Động 24/7 { display-mode: "form" }
# @markdown Nhập Server URL & Stream Key lấy từ TikTok Live Studio hoặc phần mềm phát Live:

TikTok_Server_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
TikTok_Stream_Key = "" # @param {type:"string"}

import os, time, subprocess

if not TikTok_Stream_Key.strip():
    print("⚠️ VUI LÒNG NHẬP STREAM KEY CỦA TIKTOK VÀO Ô TRÊN TRƯỚC KHI BẤM CHẠY!")
else:
    print("⏳ Đang cài đặt trình duyệt headless và màn hình ảo Xvfb 1080x1920...")
    !apt-get update -qq > /dev/null 2>&1
    !apt-get install -y -qq xvfb pulseaudio ffmpeg > /dev/null 2>&1
    
    # Dọn dẹp tiến trình phát cũ
    !pkill -9 -f Xvfb > /dev/null 2>&1
    !pkill -9 -f chrome > /dev/null 2>&1
    !pkill -9 -f ffmpeg > /dev/null 2>&1
    time.sleep(1)
    
    # Khởi tạo màn hình ảo 1080x1920 24-bit
    print("📺 Đang khởi tạo Màn hình ảo dọc 1080x1920...")
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1080x1920x24"]) 
    os.environ["DISPLAY"] = ":99"
    time.sleep(2)
    
    # Mở trình duyệt Chrome toàn màn hình vào link ?obs=1
    print("🌐 Đang mở Chánh Điện Chùa Linh Ứng trong màn hình ảo...")
    chrome_cmd = [
        "google-chrome",
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--window-size=1080,1920",
        "--start-fullscreen",
        "--kiosk",
        "--autoplay-policy=no-user-gesture-required",
        "http://localhost:3000/?obs=1"
    ]
    subprocess.Popen(chrome_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    
    # Bắt đầu đẩy luồng RTMP lên TikTok
    rtmp_destination = TikTok_Server_URL.rstrip(/) + / + TikTok_Stream_Key.lstrip(/)
    print(f"📡 ĐANG PHÁT TRỰC TIẾP LÊN TIKTOK TẠI: {TikTok_Server_URL}...")
    
    ffmpeg_cmd = (
        "ffmpeg -y -f x11grab -draw_mouse 0 -video_size 1080x1920 -framerate 30 -i :99.0 "
        "-f lavfi -i anullsrc=channel_layout=stereo:sample_rate=44100 "
        "-c:v libx264 -preset veryfast -b:v 3000k -maxrate 3500k -bufsize 6000k -pix_fmt yuv420p -g 60 "
        "-c:a aac -b:a 128k -ar 44100 "
        f"-f flv \"{rtmp_destination}\" > /tmp/ffmpeg_stream.log 2>&1 &"
    )
    os.system(ffmpeg_cmd)
    time.sleep(3)
    print("🎉 LIVESTREAM TỰ ĐỘNG ĐÃ KHỞI CHẠY THÀNH CÔNG! Kiểm tra trạng thái ở Bước 3 bên dưới.")


In [ ]:
# @title 📊 BƯỚC 3: Kiểm Tra Tốc Độ & Trạng Thái Luồng Phát TikTok
!tail -n 25 /tmp/ffmpeg_stream.log 2>/dev/null || echo "Chưa có log phát sóng."


In [ ]:
# @title ⏹️ BƯỚC 4: Dừng Toàn Bộ Hệ Thống Livestream
!pkill -9 -f ffmpeg
!pkill -9 -f chrome
!pkill -9 -f Xvfb
!pkill -9 -f "server.js"
!pkill -9 -f cloudflared
print("🛑 ĐÃ DỪNG TOÀN BỘ PHIÊN LIVESTREAM AN TOÀN!")
